# Rede Neural Perceptron — Classificação de Pureza de Óleo
**CEFET-MG Campus VIII – Varginha**  
**Disciplina:** Laboratório de Inteligência Artificial  
**Professor:** Lázaro Eduardo da Silva  

---
## Contexto do Problema
A partir da análise de um processo de destilação fracionada de petróleo, um **Perceptron de camada única** é treinado para classificar amostras de óleo em duas classes de pureza:
- **Classe C1** (convenção: `-1`)
- **Classe C2** (convenção: `+1`)

As entradas são $x_1, x_2, x_3$ (propriedades físico-químicas) lidas dos arquivos CSV `oleo_dataset.csv` (treino) e `oleo_teste.csv` (teste), acrescidas do bias $x_0 = -1$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

## Implementação da Classe Perceptron

In [ ]:
class Perceptron:
    def __init__(self, taxa_aprendizagem=0.01, max_epocas=1000, bias_input=-1):
        self.eta = taxa_aprendizagem
        self.max_epocas = max_epocas
        self.bias_input = bias_input
        self.w = None
        self.w_inicial = None
        self.historico_erros = []
        self.epocas_executadas = 0

    def ativacao(self, u):
        return np.where(u >= 0, 1.0, -1.0)

    def treinar(self, X, d, seed=None):
        if seed is not None:
            np.random.seed(seed)
        
        n_amostras, n_atributos = X.shape
        self.w = np.random.rand(n_atributos)
        self.w_inicial = self.w.copy()
        self.historico_erros = []
        self.epocas_executadas = 0
        
        for epoca in range(self.max_epocas):
            erros_na_epoca = 0
            for i in range(n_amostras):
                u = np.dot(self.w, X[i])
                y = self.ativacao(u)
                erro = d[i] - y
                if erro != 0:
                    self.w += self.eta * erro * X[i]
                    erros_na_epoca += 1
            
            self.historico_erros.append(erros_na_epoca)
            self.epocas_executadas += 1
            
            if erros_na_epoca == 0:
                break
                
        return self

    def predizer(self, X):
        u = np.dot(X, self.w)
        return self.ativacao(u)

## Carregando o Dataset de Treinamento (`oleo_dataset.csv`)

In [ ]:
# Procurar e carregar oleo_dataset.csv
caminho_treino = '../atividades/perceptron_adaline/oleo_dataset.csv'
if not os.path.exists(caminho_treino):
    caminho_treino = 'atividades/perceptron_adaline/oleo_dataset.csv'

df_treino = pd.read_csv(caminho_treino)
print("Primeiras linhas do oleo_dataset.csv:")
display(df_treino.head())

X_raw = df_treino[['x1', 'x2', 'x3']].values
d_treino = df_treino['d'].values

# Adicionar entrada bias x0 = -1
bias_col = np.full((X_raw.shape[0], 1), -1.0)
X_treino = np.hstack((bias_col, X_raw))
print(f"Conjunto de Treinamento: {X_treino.shape[0]} amostras carregadas do CSV.")

## Questões 1 e 2 — Execução e Registro dos 5 Treinamentos

In [ ]:
seeds = [1, 2, 3, 4, 5]
modelos = []
tabela_q2 = []

for idx, seed in enumerate(seeds, 1):
    p = Perceptron(taxa_aprendizagem=0.01, max_epocas=1000)
    p.treinar(X_treino, d_treino, seed=seed)
    modelos.append(p)
    
    tabela_q2.append({
        'Treinamento': f'{idx}º (T{idx})',
        'w0 (Inicial)': f'{p.w_inicial[0]:.4f}',
        'w1 (Inicial)': f'{p.w_inicial[1]:.4f}',
        'w2 (Inicial)': f'{p.w_inicial[2]:.4f}',
        'w3 (Inicial)': f'{p.w_inicial[3]:.4f}',
        'w0 (Final)': f'{p.w[0]:.4f}',
        'w1 (Final)': f'{p.w[1]:.4f}',
        'w2 (Final)': f'{p.w[2]:.4f}',
        'w3 (Final)': f'{p.w[3]:.4f}',
        'Número de Épocas': p.epocas_executadas
    })

df_q2 = pd.DataFrame(tabela_q2)
display(df_q2.style.set_caption("Resultados dos 5 Treinamentos do Perceptron"))

## Questão 3 — Classificação das Amostras de Teste (`oleo_teste.csv`)

In [ ]:
caminho_teste = '../atividades/perceptron_adaline/oleo_teste.csv'
if not os.path.exists(caminho_teste):
    caminho_teste = 'atividades/perceptron_adaline/oleo_teste.csv'

df_teste = pd.read_csv(caminho_teste)
X_teste_raw = df_teste[['x1', 'x2', 'x3']].values
X_teste = np.hstack((np.full((X_teste_raw.shape[0], 1), -1.0), X_teste_raw))

tabela_q3 = []
for i in range(len(X_teste_raw)):
    linha = {
        'Amostra': int(df_teste.iloc[i]['Amostra']),
        'x1': X_teste_raw[i, 0],
        'x2': X_teste_raw[i, 1],
        'x3': X_teste_raw[i, 2]
    }
    saidas = []
    for t, m in enumerate(modelos, 1):
        y = m.predizer(X_teste[i])
        classe_str = 'C1' if y == -1 else 'C2'
        linha[f'T{t}'] = classe_str
        saidas.append(classe_str)
    
    linha['Classe Final'] = 'Classe C1 (-1)' if saidas[0] == 'C1' else 'Classe C2 (+1)'
    tabela_q3.append(linha)

df_q3 = pd.DataFrame(tabela_q3)
display(df_q3.style.set_caption("Resultados da Classificação das 10 Amostras de Óleo (oleo_teste.csv)"))

## Evolução do Erro por Época nos Treinamentos

In [ ]:
plt.figure(figsize=(10, 5))
for i, m in enumerate(modelos, 1):
    plt.plot(m.historico_erros, label=f'T{i} ({m.epocas_executadas} épocas)', alpha=0.8)

plt.title('Evolução do Número de Erros por Época — Perceptron')
plt.xlabel('Época')
plt.ylabel('Número de Erros de Classificação')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## Questão 4 — Explicação sobre a Variação do Número de Épocas

**Pergunta:** Explique por que o número de épocas de treinamento varia a cada vez que executamos o treinamento do perceptron.

**Resposta:**  
O número de épocas varia porque em cada execução o vetor de pesos iniciais $[w_0, w_1, w_2, w_3]$ é inicializado com valores aleatórios diferentes.

1. Os pesos iniciais representam o **ponto de partida** do algoritmo no espaço de busca.
2. Pontos de partida mais próximos de um hiperplano separador válido exigem menos ajustes e convergem em menos épocas.
3. Como existem infinitos hiperplanos válidos capazes de separar dados linearmente separáveis, cada vetor inicial conduz a uma trajetória de aprendizado única.

## Questão 5 — Principal Limitação do Perceptron

**Pergunta:** Qual a principal limitação do perceptron quando aplicado em problemas de classificação de padrões?

**Resposta:**  
A principal limitação do Perceptron simples de camada única é que ele **só consegue resolver problemas cujos padrões sejam LINEARMENTE SEPARÁVEIS**.

Se o problema for não linearmente separável (como a função lógica XOR ou dados sobrepostos):
- O algoritmo do Perceptron entra em loop infinito (nunca atinge erro zero).
- Os pesos continuam oscilando indefinidamente.

Para solucionar problemas não lineares, é necessário utilizar redes neurais multicamadas (**MLP** - Multi-Layer Perceptron) com funções de ativação não lineares.